# 💳 Credit Card Fraud Detection

**Author:** Aditya Diwan  
**Email:** aditydiwan2005@gmail.com

---

## Project Overview

This notebook builds a machine learning pipeline to detect fraudulent credit card transactions.

The dataset is highly imbalanced — only ~0.17% of transactions are fraud. We handle this using **SMOTE** and train multiple models, with **XGBoost** as the primary classifier.

### Steps:
1. Import Libraries
2. Load & Explore Data (EDA)
3. Data Preprocessing
4. Handle Class Imbalance
5. Model Training
6. Model Evaluation
7. Model Interpretation (SHAP)

## 1. Import Libraries

In [ ]:
# Core
import pandas as pd
import numpy as np
from collections import Counter

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

# Preprocessing & Splitting
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler

# Imbalance Handling
from imblearn.over_sampling import SMOTE, RandomOverSampler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
from scipy.stats import ttest_ind

# Interpretability
import shap

# Settings
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print('All libraries imported successfully!')

## 2. Load & Explore Data

In [ ]:
# Load dataset
# Download from: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
data = pd.read_csv('data/creditcard.csv')

print(f'Dataset Shape: {data.shape}')
print(f'Total Transactions: {len(data):,}')
data.head()

In [ ]:
# Basic info
data.info()

In [ ]:
# Statistical summary
data.describe()

In [ ]:
# Check for missing values
missing = data.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'No missing values found.')

In [ ]:
# Missing data visualization
msno.matrix(data)
plt.title('Missing Data Matrix')
plt.show()

### 2.1 Class Distribution

In [ ]:
# Class counts
class_counts = data['Class'].value_counts()
fraud_pct = class_counts[1] / len(data) * 100
print(f"Genuine transactions : {class_counts[0]:,}")
print(f"Fraudulent transactions: {class_counts[1]:,} ({fraud_pct:.2f}%)")

# Donut chart
fig, ax = plt.subplots(figsize=(6, 6))
colors = ['#66b3ff', '#ff9999']
ax.pie(class_counts, labels=['Genuine', 'Fraud'], colors=colors,
       startangle=90, counterclock=False,
       wedgeprops=dict(width=0.4), autopct='%1.2f%%')
centre_circle = plt.Circle((0, 0), 0.6, color='white')
ax.add_artist(centre_circle)
plt.title('Class Distribution', fontsize=14)
plt.show()

### 2.2 Transaction Amount Distribution

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

sns.histplot(data[data['Class'] == 0]['Amount'], ax=axes[0], color='steelblue', kde=True)
axes[0].set_title('Amount Distribution — Genuine Transactions')
axes[0].set_xlabel('Transaction Amount')

sns.histplot(data[data['Class'] == 1]['Amount'], ax=axes[1], color='tomato', kde=True)
axes[1].set_title('Amount Distribution — Fraudulent Transactions')
axes[1].set_xlabel('Transaction Amount')

plt.tight_layout()
plt.show()

### 2.3 Time Distribution

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

sns.histplot(data[data['Class'] == 0]['Time'], ax=axes[0], color='steelblue', kde=True)
axes[0].set_title('Time Distribution — Genuine Transactions')

sns.histplot(data[data['Class'] == 1]['Time'], ax=axes[1], color='tomato', kde=True)
axes[1].set_title('Time Distribution — Fraudulent Transactions')

plt.tight_layout()
plt.show()

### 2.4 Feature Distributions (V1–V5)

In [ ]:
features = ['V1', 'V2', 'V3', 'V4', 'V5']
fig, axes = plt.subplots(len(features), 1, figsize=(12, 4 * len(features)))

for ax, feature in zip(axes, features):
    sns.histplot(data[data['Class'] == 0][feature], ax=ax, color='steelblue', kde=True, label='Genuine', alpha=0.6)
    sns.histplot(data[data['Class'] == 1][feature], ax=ax, color='tomato', kde=True, label='Fraud', alpha=0.6)
    ax.set_title(f'{feature} Distribution by Class')
    ax.legend()

plt.tight_layout()
plt.show()

### 2.5 Correlation Heatmap

In [ ]:
plt.figure(figsize=(14, 10))
corr = data.corr()
sns.heatmap(corr, cmap='coolwarm', linewidths=0.5, annot=False)
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

### 2.6 Density Plots

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

sns.kdeplot(data[data['Class'] == 0]['Amount'], fill=True, color='steelblue', ax=axes[0], label='Genuine')
axes[0].set_title('Density — Genuine Transaction Amounts')
axes[0].set_xlabel('Transaction Amount')
axes[0].set_ylabel('Density')

sns.kdeplot(data[data['Class'] == 1]['Amount'], fill=True, color='tomato', ax=axes[1], label='Fraud')
axes[1].set_title('Density — Fraudulent Transaction Amounts')
axes[1].set_xlabel('Transaction Amount')
axes[1].set_ylabel('Density')

plt.tight_layout()
plt.show()

### 2.7 Statistical Test — Are Fraud and Genuine Amounts Different?

In [ ]:
normal_amounts = data[data['Class'] == 0]['Amount']
fraud_amounts  = data[data['Class'] == 1]['Amount']

t_stat, p_val = ttest_ind(normal_amounts, fraud_amounts)

print(f'T-statistic : {t_stat:.4f}')
print(f'P-value     : {p_val:.6f}')
print()
if p_val < 0.05:
    print('Result: Statistically significant difference between fraud and genuine transaction amounts (p < 0.05).')
else:
    print('Result: No statistically significant difference found.')

## 3. Data Preprocessing

In [ ]:
# Separate features and target
X = data.drop('Class', axis=1)
y = data['Class']

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set  : {X_train.shape[0]:,} samples')
print(f'Test set      : {X_test.shape[0]:,} samples')

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Scaling applied successfully.')

## 4. Handle Class Imbalance

The dataset has ~0.17% fraud cases. We use **SMOTE** (Synthetic Minority Oversampling Technique) to balance the training data.

In [ ]:
# Apply SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f'Before SMOTE: {Counter(y_train)}')
print(f'After SMOTE : {Counter(y_resampled)}')

In [ ]:
# Visualize class distribution before and after SMOTE
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

before = Counter(y_train)
after  = Counter(y_resampled)

axes[0].bar(['Genuine', 'Fraud'], [before[0], before[1]], color=['steelblue', 'tomato'])
axes[0].set_title('Class Distribution — Before SMOTE')
axes[0].set_ylabel('Count')

axes[1].bar(['Genuine', 'Fraud'], [after[0], after[1]], color=['steelblue', 'tomato'])
axes[1].set_title('Class Distribution — After SMOTE')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 5. Model Training

We train three models:
1. **Isolation Forest** — unsupervised anomaly detection
2. **Logistic Regression** — with hyperparameter tuning (GridSearchCV)
3. **XGBoost** — gradient boosting on SMOTE-balanced data

### 5.1 Isolation Forest (Anomaly Detection)

In [ ]:
iso_forest = IsolationForest(contamination=0.01, random_state=42)
iso_forest.fit(X_train_scaled)

# Isolation Forest returns -1 (anomaly) and 1 (normal)
# Convert to 1 (fraud) and 0 (genuine) to match dataset labels
y_pred_iso = iso_forest.predict(X_test_scaled)
y_pred_iso = [1 if p == -1 else 0 for p in y_pred_iso]

print('Isolation Forest — Classification Report')
print('=' * 50)
print(classification_report(y_test, y_pred_iso, target_names=['Genuine', 'Fraud']))

### 5.2 Logistic Regression with Hyperparameter Tuning

In [ ]:
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10], 'penalty': ['l2']}

grid_search = GridSearchCV(
    LogisticRegression(solver='liblinear', max_iter=1000),
    param_grid, cv=5, scoring='f1', n_jobs=-1
)
grid_search.fit(X_resampled, y_resampled)

print(f'Best Parameters: {grid_search.best_params_}')

best_lr = grid_search.best_estimator_
y_pred_lr = best_lr.predict(X_test_scaled)

print()
print('Logistic Regression — Classification Report')
print('=' * 50)
print(classification_report(y_test, y_pred_lr, target_names=['Genuine', 'Fraud']))

### 5.3 XGBoost Classifier

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)
xgb_model.fit(X_resampled, y_resampled)
y_pred_xgb = xgb_model.predict(X_test_scaled)

print('XGBoost — Classification Report')
print('=' * 50)
print(classification_report(y_test, y_pred_xgb, target_names=['Genuine', 'Fraud']))

## 6. Model Evaluation

In [ ]:
# Compare all models side-by-side
models = {
    'Isolation Forest': y_pred_iso,
    'Logistic Regression': y_pred_lr,
    'XGBoost': y_pred_xgb
}

results = []
for name, y_pred in models.items():
    results.append({
        'Model': name,
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall':    recall_score(y_test, y_pred, zero_division=0),
        'F1 Score':  f1_score(y_test, y_pred, zero_division=0)
    })

results_df = pd.DataFrame(results).set_index('Model')
print(results_df.round(4))

In [ ]:
# Bar chart comparison
results_df.plot(kind='bar', figsize=(10, 6), colormap='Set2', edgecolor='black')
plt.title('Model Comparison — Precision, Recall, F1 Score', fontsize=14)
plt.ylabel('Score')
plt.ylim(0, 1.1)
plt.xticks(rotation=15)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 7. Model Interpretation (SHAP)

SHAP (SHapley Additive Explanations) shows which features contributed most to the XGBoost model's predictions.

In [ ]:
# Use a sample for speed (SHAP can be slow on full dataset)
X_test_df = pd.DataFrame(X_test_scaled, columns=X.columns)
sample = X_test_df.sample(500, random_state=42)

explainer   = shap.Explainer(xgb_model)
shap_values = explainer(sample)

# Summary bar plot — feature importance
shap.summary_plot(shap_values, sample, plot_type='bar')

In [ ]:
# Detailed SHAP beeswarm plot
shap.summary_plot(shap_values, sample)

---

## Summary

| Step | Technique | Outcome |
|---|---|---|
| EDA | Distribution plots, heatmaps, t-test | Understood data patterns |
| Preprocessing | StandardScaler | Normalized features |
| Imbalance | SMOTE | Balanced training set |
| Model 1 | Isolation Forest | Anomaly detection baseline |
| Model 2 | Logistic Regression + GridSearchCV | Tuned linear classifier |
| Model 3 | XGBoost | Best overall performance |
| Interpretation | SHAP values | Feature importance explained |

---

**Author:** Aditya Diwan | aditydiwan2005@gmail.com